# NSCLC scRNA-seq (P1–P4): merge samples, QC, and clustering before integration (Seurat / R)

This notebook applies the PBMC stage-1 workflow (`analysis_stage1.ipynb`) to the NSCLC samples P1–P4 and stops just before Harmony integration:

1. Load each sample folder (gene expression, TotalSeq antibodies and CellPlex tags) and **merge them into one Seurat object**, which is saved
2. QC metrics (mitochondrial and ribosomal content, genes and UMIs per cell), doublet detection with scDblFinder, and QC filtering
3. Normalisation, highly variable genes (HVGs), PCA
4. Neighbour graph, Louvain clustering at two resolutions and UMAP, **without** Harmony
5. Every plot is coloured by **sample**, and the sample make-up of each cluster is tabulated. This is the batch effect Harmony will correct.

**All inputs and outputs are stored in `E:/DeCodeLife_scRNAseq`:**

| | Location |
|---|---|
| Input: one 10x folder per sample | `E:/DeCodeLife_scRNAseq/NSCLC_7pt/P1/` … `P4/` (`barcodes.tsv.gz`, `features.tsv.gz`, `matrix.mtx.gz`) |
| Output: merged raw object | `E:/DeCodeLife_scRNAseq/data/nsclc7pt_merged.rds` |
| Output: processed object (pre-Harmony) | `E:/DeCodeLife_scRNAseq/data/nsclc7pt_preInt.rds` |
| Output: figures | `E:/DeCodeLife_scRNAseq/data/figures/NSCLC_7pt/` |
| Output: tables | `E:/DeCodeLife_scRNAseq/data/tables/NSCLC_7pt/` |

All results (figures, tables and `.rds` objects) are written only to `data/`. `data/` and the input data folder are listed in `.gitignore`, so neither is pushed to GitHub.

**Requirements:** R >= 4.3, Seurat >= 5.0, ggplot2, patchwork, scDblFinder and SingleCellExperiment (Bioconductor), and the Jupyter R kernel (IRkernel).

### About the dataset

The 7 NSCLC donors were pooled with 10x CellPlex and processed with Cell Ranger 6.1.2 `multi`. Each folder P1–P4 is one donor's per-sample output (`sample_filtered_feature_bc_matrix`). The dominant CellPlex tag of the cells is CMO301 in P1, CMO302 in P2, CMO303 in P3 and CMO304 in P4. All four folders share the same feature list:

| Feature type | Features | Used for |
|---|---|---|
| Gene Expression | 36,601 genes | `RNA` assay: all of the analysis below |
| Antibody Capture | 9 TotalSeq antibodies: CD3, CD4, CD8, CD11c, CD14, CD16, CD19, CD56, CD45 | `ADT` assay: protein levels on the UMAP |
| Multiplexing Capture | 12 CellPlex tags: CMO301 to CMO312 | `CMO` assay: confirms which donor each folder holds |

The folder name, not the tag, is the sample label (`sample` = P1 to P4). This label is the batch variable Harmony will correct for.

## Setup

In [ ]:
# Run once if the packages are missing:
# install.packages(c("Seurat", "ggplot2", "patchwork"))

suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
  library(ggplot2)
  library(patchwork)
})

seed <- 198716                       # same seed as the PBMC stage-1 notebook
set.seed(seed)
options(repr.plot.width = 10, repr.plot.height = 6, repr.plot.res = 120)  # inline plot size in Jupyter
packageVersion("Seurat")

In [ ]:
# ---- All inputs and outputs live in this folder ----
base_dir    <- "E:/DeCodeLife_scRNAseq"
data_root   <- file.path(base_dir, "NSCLC_7pt")           # 10x input (git-ignored)
results_dir <- file.path(base_dir, "data")                # every result goes here; data/ is git-ignored, never pushed to GitHub
fig_dir     <- file.path(results_dir, "figures", "NSCLC_7pt")
table_dir   <- file.path(results_dir, "tables",  "NSCLC_7pt")
merged_rds  <- file.path(results_dir, "nsclc7pt_merged.rds")    # merged raw object (before QC)
rds_file    <- file.path(results_dir, "nsclc7pt_preInt.rds")    # processed object (before Harmony)

# One 10x folder per sample; the folder name becomes the sample label. Add more names here (e.g. "P5").
sample_names <- c("P1", "P2", "P3", "P4")
sample_dirs  <- setNames(file.path(data_root, sample_names), sample_names)

required_files <- c("barcodes.tsv.gz", "features.tsv.gz", "matrix.mtx.gz")
for (s in names(sample_dirs)) {
  missing_files <- required_files[!file.exists(file.path(sample_dirs[[s]], required_files))]
  if (length(missing_files) > 0) {
    stop("Sample ", s, ": missing ", paste(missing_files, collapse = ", "), " in ", sample_dirs[[s]])
  }
}

dir.create(fig_dir,   showWarnings = FALSE, recursive = TRUE)
dir.create(table_dir, showWarnings = FALSE, recursive = TRUE)

# Show a plot inline and save a copy in fig_dir
save_plot <- function(p, filename, width = 10, height = 6) {
  ggsave(file.path(fig_dir, filename), plot = p, width = width, height = height, dpi = 300, bg = "white")
  p
}

# Fraction of total variance explained by each PC (Scanpy: adata.uns['pca']['variance_ratio'])
pca_variance_ratio <- function(obj, reduction = "pca") {
  stdev <- Stdev(obj, reduction = reduction)
  total <- obj[[reduction]]@misc$total.variance
  data.frame(PC = seq_along(stdev), variance_ratio = stdev^2 / total)
}

sample_dirs

#### A: Load the samples and merge them

Each folder becomes its own Seurat object with three assays (`RNA`, `ADT`, `CMO`), and the four objects are merged. `merge()` prefixes every cell name with its sample (`P1_AAACCCAG…-1`). In Seurat 5 it also keeps each sample's counts in a separate layer (`counts.P1`, `counts.P2`, …), so the layers are joined back into one `counts` layer per assay.

In [ ]:
# All samples must share the same reference genes and antibody panel
read_features <- function(sample_dir) {
  read.delim(file.path(sample_dir, "features.tsv.gz"), header = FALSE,
             col.names = c("gene_ids", "gene_symbol", "feature_types"))
}
feature_tables <- lapply(sample_dirs, read_features)
same_features  <- vapply(feature_tables, identical, logical(1), feature_tables[[1]])
if (!all(same_features)) stop("features.tsv.gz differs for: ", paste(names(which(!same_features)), collapse = ", "))
table(feature_tables[[1]]$feature_types)

In [ ]:
# Scanpy: sc.read_10x_mtx(path, var_names='gene_symbols', gex_only=False)
# Read10X returns one matrix per feature type: "Gene Expression", "Antibody Capture", "Multiplexing Capture"
read_10x_sample <- function(sample_dir, sample_name) {
  mats     <- Read10X(data.dir = sample_dir, gene.column = 2)            # gene symbols as row names
  if (!is.list(mats)) mats <- list("Gene Expression" = mats)
  features <- read_features(sample_dir)

  obj <- CreateSeuratObject(counts = mats[["Gene Expression"]], project = sample_name,
                            min.cells = 0, min.features = 0)
  obj$sample <- sample_name

  # TotalSeq antibodies. Read10X makes names unique across all feature types, so antibodies named like a gene
  # (CD4, CD14, CD19) come back as "CD4.1" etc.; restore the panel names (ADT is its own assay).
  if (!is.null(mats[["Antibody Capture"]])) {
    adt <- mats[["Antibody Capture"]]
    rownames(adt) <- features$gene_symbol[features$feature_types == "Antibody Capture"]
    obj[["ADT"]] <- CreateAssay5Object(counts = adt)
  }

  # CellPlex tags: dominant tag per cell, and its share of the cell's tag counts (how clean the call is)
  if (!is.null(mats[["Multiplexing Capture"]])) {
    cmo <- mats[["Multiplexing Capture"]]
    obj[["CMO"]] <- CreateAssay5Object(counts = cmo)
    cmo_mat <- t(as.matrix(cmo))                                         # cells x tags (small)
    obj$cmo_tag      <- colnames(cmo_mat)[max.col(cmo_mat, ties.method = "first")]
    obj$cmo_top_frac <- apply(cmo_mat, 1, max) / pmax(rowSums(cmo_mat), 1)
  }
  obj
}

sample_objs <- Map(read_10x_sample, sample_dirs, names(sample_dirs))
sample_objs

In [ ]:
# Merge all samples into one object; cell names get the sample as prefix (P1_<barcode>, ...)
if (length(sample_objs) < 2) stop("merge() needs at least two samples")
nsclc_merged <- merge(x = sample_objs[[1]], y = sample_objs[-1],
                      add.cell.ids = names(sample_objs), project = "NSCLC_7pt")
rm(sample_objs)

Layers(nsclc_merged[["RNA"]])     # one counts layer per sample: counts.P1, counts.P2, ...

# Join the per-sample layers into a single "counts" layer in every assay
for (assay in Assays(nsclc_merged)) {
  nsclc_merged <- JoinLayers(nsclc_merged, assay = assay)
}
DefaultAssay(nsclc_merged) <- "RNA"
nsclc_merged$sample <- factor(nsclc_merged$sample, levels = names(sample_dirs))

Layers(nsclc_merged[["RNA"]])
nsclc_merged

In [ ]:
cat("Number of cells =", ncol(nsclc_merged), "\n")
cat("Number of genes =", nrow(nsclc_merged), "\n")

# Scanpy: var_names_make_unique() / obs_names_make_unique()
# Read10X() already makes duplicated gene symbols unique (e.g. "GENE", "GENE.1"); confirm nothing is duplicated
c(duplicated_genes = sum(duplicated(rownames(nsclc_merged))),
  duplicated_cells = sum(duplicated(colnames(nsclc_merged))))

##### Samples

Each sample should hold one donor, so nearly all cells in a sample should share one dominant CellPlex tag. Every tag in use shows a large total in its own sample; the others only carry ambient counts, a few orders of magnitude lower.

In [ ]:
table(sample = nsclc_merged$sample)
table(sample = nsclc_merged$sample, cmo_tag = nsclc_merged$cmo_tag)
tapply(nsclc_merged$cmo_top_frac, nsclc_merged$sample, summary)

In [ ]:
# Total counts per CellPlex tag in each sample
cmo_counts <- LayerData(nsclc_merged, assay = "CMO", layer = "counts")
cmo_totals <- do.call(rbind, lapply(levels(nsclc_merged$sample), function(s) {
  cells_s <- colnames(nsclc_merged)[nsclc_merged$sample == s]
  data.frame(sample = s, tag = rownames(cmo_counts),
             total_counts = Matrix::rowSums(cmo_counts[, cells_s, drop = FALSE]))
}))
rm(cmo_counts)

options(repr.plot.width = 12, repr.plot.height = 6)
p <- ggplot(cmo_totals, aes(x = tag, y = total_counts + 1)) +
  geom_col(fill = "steelblue") +
  scale_y_log10() +
  facet_wrap(~ sample, ncol = 2) +
  labs(title = "Total counts per CellPlex tag, by sample", x = NULL, y = "Total counts + 1 (log scale)") +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))
save_plot(p, "01_cmo_tag_totals_by_sample.png", width = 12, height = 6)

In [ ]:
# Save the merged raw object (all genes and cells, before QC)
saveRDS(nsclc_merged, file = merged_rds)

#### B: QC metrics

In [ ]:
# Scanpy: sc.pp.calculate_qc_metrics(qc_vars=['mt', 'ribo'], percent_top=False, log1p=False)
# nFeature_RNA (= n_genes_by_counts) and nCount_RNA (= total_counts) are added by CreateSeuratObject
nsclc_merged[["percent.mt"]]   <- PercentageFeatureSet(nsclc_merged, pattern = "^MT-")
nsclc_merged[["percent.ribo"]] <- PercentageFeatureSet(nsclc_merged, pattern = "^RP[SL]")   # genes starting with RPL / RPS

raw_counts <- LayerData(nsclc_merged, assay = "RNA", layer = "counts")
mt_genes   <- grepl("^MT-",    rownames(raw_counts))
ribo_genes <- grepl("^RP[SL]", rownames(raw_counts))
nsclc_merged$total_counts_mt   <- Matrix::colSums(raw_counts[mt_genes, , drop = FALSE])
nsclc_merged$total_counts_ribo <- Matrix::colSums(raw_counts[ribo_genes, , drop = FALSE])

# Ensembl IDs and feature types (Scanpy keeps these in adata.var). Gene expression rows come first in
# features.tsv, so make.unique() of their symbols reproduces the row names Read10X gave the RNA assay
# (Seurat also replaces "_" with "-" in feature names).
gene_info <- feature_tables[[1]][feature_tables[[1]]$feature_types == "Gene Expression", ]
gene_info <- gene_info[match(rownames(raw_counts), gsub("_", "-", make.unique(gene_info$gene_symbol))), ]
stopifnot(!anyNA(gene_info$gene_ids))

# Gene-level QC metrics (Scanpy: adata.var)
gene_qc <- data.frame(
  gene_ids          = gene_info$gene_ids,
  feature_types     = gene_info$feature_types,
  mt                = mt_genes,
  ribo              = ribo_genes,
  n_cells_by_counts = Matrix::rowSums(raw_counts > 0),
  mean_counts       = Matrix::rowMeans(raw_counts),
  total_counts      = Matrix::rowSums(raw_counts),
  row.names         = rownames(raw_counts)
)
gene_qc$pct_dropout_by_counts <- 100 * (1 - gene_qc$n_cells_by_counts / ncol(raw_counts))
nsclc_merged[["RNA"]] <- AddMetaData(nsclc_merged[["RNA"]], metadata = gene_qc)
rm(raw_counts)

head(nsclc_merged@meta.data)

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5)
qc_features <- c("nFeature_RNA", "nCount_RNA", "percent.mt", "percent.ribo")

p <- VlnPlot(nsclc_merged, features = qc_features, group.by = "sample", ncol = 4, pt.size = 0.1) & NoLegend()
save_plot(p, "02_qc_violin_raw.png", width = 14, height = 5)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4)
p <- ggplot(nsclc_merged@meta.data, aes(x = percent.mt, fill = sample)) +
  geom_histogram(bins = 100) +
  labs(title = "Percent mitochondrial counts", x = "percent.mt", y = "Number of cells") +
  theme_classic()
save_plot(p, "03_hist_percent_mt.png", width = 7, height = 4)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4)
p <- ggplot(nsclc_merged@meta.data, aes(x = percent.ribo, fill = sample)) +
  geom_histogram(bins = 100) +
  labs(title = "Percent ribosomal counts", x = "percent.ribo", y = "Number of cells") +
  theme_classic()
save_plot(p, "04_hist_percent_ribo.png", width = 7, height = 4)

##### Doublet detection (scDblFinder)

A doublet is a droplet that captured two cells. It looks like one cell with about twice the UMIs and genes, and when the two cells are of different types (e.g. a tumour cell and a macrophage) it can form a fake "intermediate" cluster.

`scDblFinder` (Bioconductor; the Scanpy counterpart is `sc.pp.scrublet`) builds artificial doublets by adding up random pairs of cells, then trains a classifier to tell them apart from the real cells. Each cell gets a `scDblFinder.score` (0 to 1) and a `scDblFinder.class` (`singlet` or `doublet`).

**Why it runs per sample:** all donors were loaded into the same 10x capture, so doublets could form between donors. However, Cell Ranger `multi` has already removed droplets carrying two different CellPlex tags from these per-sample folders. The doublets left are two cells from the **same** donor, so scDblFinder builds its artificial doublets within each sample (`samples = sample`). Its default expected rate (0.8% per 1,000 cells in each sample, about 1 to 1.5% here) is a reasonable expectation for these same-donor doublets. The two methods complement each other: CellPlex finds cross-donor doublets, and scDblFinder mainly finds doublets of two different cell types.

As the scDblFinder guidelines recommend, it runs **before** the QC filter, on every barcode with at least 200 genes. The doublets are then removed together with the QC thresholds below.

SoupX is a different tool: it removes ambient RNA, not doublets, and it needs Cell Ranger's `raw_feature_bc_matrix` (all droplets, including empty ones), which is not part of this dataset.

In [ ]:
# Run once if the packages are missing:
# install.packages("BiocManager"); BiocManager::install(c("scDblFinder", "SingleCellExperiment"))
# Functions are called with pkg:: so the Bioconductor packages are not attached (they would mask some base R names)

# Barcodes with < 200 genes carry too little signal to score; they fail the QC filter anyway
dbl_cells <- colnames(nsclc_merged)[nsclc_merged$nFeature_RNA >= 200]
sce <- SingleCellExperiment::SingleCellExperiment(
  assays = list(counts = LayerData(nsclc_merged, assay = "RNA", layer = "counts")[, dbl_cells])
)

# one scDblFinder run per sample (artificial doublets are made within each sample)
set.seed(seed)
sce <- scDblFinder::scDblFinder(sce, samples = as.character(nsclc_merged@meta.data[dbl_cells, "sample"]),
                                BPPARAM = BiocParallel::SerialParam(RNGseed = seed))

dbl_calls <- data.frame(
  scDblFinder.class = factor(as.character(sce$scDblFinder.class), levels = c("singlet", "doublet")),
  scDblFinder.score = sce$scDblFinder.score,
  row.names         = colnames(sce)
)
nsclc_merged <- AddMetaData(nsclc_merged, metadata = dbl_calls)   # barcodes with < 200 genes get NA
rm(sce, dbl_calls)

table(sample = nsclc_merged$sample, scDblFinder.class = nsclc_merged$scDblFinder.class, useNA = "ifany")

In [ ]:
# Doublets should have more UMIs and genes than singlets, in every sample
dbl_meta <- nsclc_merged@meta.data[!is.na(nsclc_merged$scDblFinder.class), ]

dbl_violin <- function(feature) {
  ggplot(dbl_meta, aes(x = scDblFinder.class, y = .data[[feature]], fill = scDblFinder.class)) +
    geom_violin(scale = "width") +
    scale_y_log10() +
    facet_wrap(~ sample, nrow = 1) +
    labs(x = NULL, y = feature) +
    theme_bw() +
    theme(legend.position = "none")
}
p_score <- ggplot(dbl_meta, aes(x = scDblFinder.score, fill = scDblFinder.class)) +
  geom_histogram(bins = 50) +
  facet_wrap(~ sample, nrow = 1) +
  labs(x = "scDblFinder.score", y = "Number of cells", fill = NULL) +
  theme_bw()

options(repr.plot.width = 14, repr.plot.height = 11)
p <- dbl_violin("nCount_RNA") / dbl_violin("nFeature_RNA") / p_score
save_plot(p, "04b_doublets_scDblFinder.png", width = 14, height = 11)

##### QC thresholds

As in stage 1, the thresholds depend on the dataset. The PBMC cut-offs (`percent.mt < 10`, 200 to 4,000 genes) are too strict for dissociated tumour:

- Tumour and epithelial cells express many more genes than resting blood cells.
- Tissue dissociation stresses cells, so mitochondrial content is higher than in blood. A 15 to 20% cut-off is common for tumour tissue.
- The scDblFinder doublets are removed as well. `max_genes` stays as a backstop for extreme cells.

The same thresholds are applied to every sample. Compare them with the per-sample violin plots and histograms above, and adjust them if one sample looks very different.

In [ ]:
max_pct_mt <- 20     # remove dying / damaged cells
min_genes  <- 200    # remove empty droplets and debris
max_genes  <- 8000   # remove cells with unusually many genes (backstop for doublets scDblFinder misses)

qc_thresholds <- (nsclc_merged$percent.mt < max_pct_mt) &
                 (nsclc_merged$nFeature_RNA >= min_genes & nsclc_merged$nFeature_RNA <= max_genes)
is_singlet    <- nsclc_merged$scDblFinder.class %in% "singlet"   # FALSE for doublets and unscored barcodes (< 200 genes)

# How many doublets do the thresholds catch on their own?
table(passes_thresholds = qc_thresholds, scDblFinder.class = nsclc_merged$scDblFinder.class, useNA = "ifany")

qc_pass <- qc_thresholds & is_singlet
table(sample = nsclc_merged$sample, qc_pass = qc_pass)

In [ ]:
nsclc_filtered <- subset(nsclc_merged, cells = colnames(nsclc_merged)[qc_pass])
nsclc_filtered

In [ ]:
# Scanpy: sc.pp.filter_cells(min_genes=200) then sc.pp.filter_genes(min_cells=5)
min_cells   <- 5
filt_counts <- LayerData(nsclc_filtered, assay = "RNA", layer = "counts")
cells_keep  <- colnames(filt_counts)[Matrix::colSums(filt_counts > 0) >= min_genes]           # cell expresses >= 200 genes
n_cells     <- Matrix::rowSums(filt_counts[, cells_keep, drop = FALSE] > 0)
genes_keep  <- names(n_cells)[n_cells >= min_cells]                                         # gene detected in >= 5 cells
rm(filt_counts)

nsclc_filtered <- subset(nsclc_filtered, cells = cells_keep)

# Filter genes in the RNA assay only. subset(object, features = ...) would apply the gene list to every assay:
# the CMO assay would be dropped and ADT cut down to CD4 / CD14 / CD19 (antibodies named like genes).
# Replacing the assay triggers an expected "Different cells and/or features" warning, which is silenced here.
suppressWarnings(nsclc_filtered[["RNA"]] <- subset(nsclc_filtered[["RNA"]], features = genes_keep))
nsclc_filtered$n_genes <- nsclc_filtered$nFeature_RNA
nsclc_filtered[["RNA"]] <- AddMetaData(nsclc_filtered[["RNA"]],
                                       metadata = data.frame(n_cells = n_cells[genes_keep], row.names = genes_keep))

nsclc_filtered
table(sample = nsclc_filtered$sample)

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5)
p <- VlnPlot(nsclc_filtered, features = qc_features, group.by = "sample", ncol = 4, pt.size = 0.1) & NoLegend()
save_plot(p, "05_qc_violin_filtered.png", width = 14, height = 5)

##### C

Scaling and normalisation

In [ ]:
# Scanpy: sc.pp.normalize_total(target_sum=1e4) + sc.pp.log1p  ->  log1p(counts / total_counts * 1e4)
# Seurat writes the result to the "data" layer and keeps the raw counts in the "counts" layer
nsclc_filtered <- NormalizeData(nsclc_filtered, assay = "RNA", normalization.method = "LogNormalize", scale.factor = 1e4)

# Antibodies: centred log-ratio per cell (standard for CITE-seq / TotalSeq data); used only for plots here
nsclc_filtered <- NormalizeData(nsclc_filtered, assay = "ADT", normalization.method = "CLR", margin = 2)
nsclc_filtered

**HVGs:** 2,000 (the Seurat default) is also common for heterogeneous tumour samples.

In [ ]:
n_hvg <- 2000

nsclc_filtered <- FindVariableFeatures(nsclc_filtered, selection.method = "vst", nfeatures = n_hvg)
hvg_genes <- VariableFeatures(nsclc_filtered)        # ordered: most variable gene first

hvg_df <- HVFInfo(nsclc_filtered, method = "vst")
hvg_df$gene_name            <- rownames(hvg_df)
hvg_df$highly_variable      <- hvg_df$gene_name %in% hvg_genes
hvg_df$highly_variable_rank <- match(hvg_df$gene_name, hvg_genes)
hvg_df <- hvg_df[hvg_df$highly_variable, ]          # keep only the HVGs (genome order)

write.csv(hvg_df, file.path(table_dir, "nsclc7pt_hvg.csv"), row.names = FALSE)
head(hvg_df)

In [ ]:
top10_hvgs <- hvg_df$gene_name[order(hvg_df$highly_variable_rank)][1:10]
top10_hvgs

In [ ]:
# Seurat's standard HVG plot: mean vs standardised variance, top 10 labelled
options(repr.plot.width = 8, repr.plot.height = 5)
p <- LabelPoints(plot = VariableFeaturePlot(nsclc_filtered), points = top10_hvgs, repel = TRUE)
save_plot(p, "06_variable_features.png", width = 8, height = 5)

In [ ]:
# Violin plot of the top 10 HVGs in one panel (Scanpy: sc.pl.violin(..., multi_panel=False))
expr_mat  <- as.matrix(LayerData(nsclc_filtered, assay = "RNA", layer = "data")[top10_hvgs, ])
expr_long <- data.frame(
  gene       = factor(rep(top10_hvgs, times = ncol(expr_mat)), levels = top10_hvgs),
  expression = as.vector(expr_mat)
)
rm(expr_mat)

options(repr.plot.width = 10, repr.plot.height = 5)
p <- ggplot(expr_long, aes(x = gene, y = expression, fill = gene)) +
  geom_violin(scale = "width", trim = TRUE) +
  geom_jitter(width = 0.2, size = 0.1, alpha = 0.3) +
  labs(x = "Genes", y = "Expression (normalised log1p values)", title = "Violin Plot of Top 10 HVGs") +
  theme_classic() +
  theme(axis.text.x  = element_text(angle = 45, hjust = 1),
        axis.title   = element_text(size = 12),
        plot.title   = element_text(size = 14),
        legend.position = "none")
save_plot(p, "07_violin_top10_hvgs.png", width = 10, height = 5)

##### Dimensionality reduction (without Harmony)

As in stage 1, genes are only mean-centred before PCA (`ScaleData(do.scale = FALSE)`, which matches Scanpy's `sc.pp.pca`), and PCA uses the HVGs only.

One difference from stage 1: stage 1 cut the object down to the 1,000 HVGs, while here **all genes are kept**. PCA, clusters and UMAP come out the same, because PCA reads only the HVGs either way. Keeping every gene lets marker detection and cell-type annotation after Harmony use the whole transcriptome.

In [ ]:
nsclc_filtered <- ScaleData(nsclc_filtered, features = hvg_genes, do.center = TRUE, do.scale = FALSE)
nsclc_filtered <- RunPCA(nsclc_filtered, features = hvg_genes, npcs = 50, verbose = FALSE)

In [ ]:
# PCA coloured by sample, plus two compartment markers: PTPRC (CD45, immune) and EPCAM (epithelial / tumour)
var_ratio <- pca_variance_ratio(nsclc_filtered)
pc_labs   <- labs(x = sprintf("PC1 (%.1f%%)", 100 * var_ratio$variance_ratio[1]),
                  y = sprintf("PC2 (%.1f%%)", 100 * var_ratio$variance_ratio[2]))
pca_genes <- intersect(c("PTPRC", "EPCAM"), rownames(nsclc_filtered))

pca_plots <- c(list(DimPlot(nsclc_filtered, reduction = "pca", group.by = "sample", shuffle = TRUE, seed = seed) + pc_labs),
               lapply(pca_genes, function(g) FeaturePlot(nsclc_filtered, features = g, reduction = "pca") + pc_labs))

options(repr.plot.width = 6 * length(pca_plots), repr.plot.height = 5)
p <- wrap_plots(pca_plots, nrow = 1)
save_plot(p, "08_pca_sample_markers.png", width = 6 * length(pca_plots), height = 5)

In [ ]:
# Scanpy: sc.pl.pca_variance_ratio(n_pcs=50); ElbowPlot is the usual Seurat equivalent (standard deviation per PC)
options(repr.plot.width = 12, repr.plot.height = 4.5)
p <- ggplot(var_ratio, aes(x = PC, y = variance_ratio)) +
  geom_point() +
  labs(title = "PCA variance ratio", x = "Principal component", y = "Variance ratio") +
  theme_classic()
p <- p | ElbowPlot(nsclc_filtered, ndims = 50)
save_plot(p, "09_pca_variance_ratio.png", width = 12, height = 4.5)

In [ ]:
# Scanpy: sc.pl.pca_loadings(components=(1, 2), include_lowest=True)
# balanced = TRUE shows the most positive and the most negative loadings
options(repr.plot.width = 10, repr.plot.height = 6)
p <- VizDimLoadings(nsclc_filtered, dims = 1:2, reduction = "pca", nfeatures = 30, balanced = TRUE)
save_plot(p, "10_pca_loadings_PC1_PC2.png", width = 10, height = 6)

##### Number of PCs

Stage 1 read the elbow off the PBMC plot (7 PCs). The NSCLC data have their own elbow, so the cell below picks the number of PCs with a common rule of thumb (from the HBC single-cell training). It uses each PC's share of the total standard deviation, and takes whichever of these two PCs comes first:

- the PC where the cumulative share passes 90% while the PC on its own adds less than 5%
- the PC just after the last drop of more than 0.1 percentage points between consecutive PCs

Compare the result with the elbow plot above, and set `n_pcs` by hand if you disagree.

In [ ]:
stdev <- Stdev(nsclc_filtered, reduction = "pca")
pct   <- 100 * stdev / sum(stdev)
cumu  <- cumsum(pct)

pc_cumulative <- which(cumu > 90 & pct < 5)[1]
pc_drop       <- tail(which(diff(pct) < -0.1), 1) + 1
n_pcs <- min(pc_cumulative, pc_drop, na.rm = TRUE)
# n_pcs <- 10   # uncomment to set it by hand after looking at the elbow plot

nsclc_filtered@misc$n_pcs <- n_pcs   # stored in the object for the Harmony notebook
c(pc_cumulative = pc_cumulative, pc_drop = pc_drop, n_pcs = n_pcs)

#### E: Clustering and UMAP without Harmony

Same settings as stage 1: neighbour graph with `k = 10`, Louvain at resolutions 0.5 and 0.7, UMAP with `min.dist = 0.5`. All of them use the uncorrected PCA. Results are stored as `preInt_louvain_0.5`, `preInt_louvain_0.7` and the `umap.preInt` reduction, so the Harmony results can be added alongside them without overwriting anything.

In [ ]:
# Scanpy: sc.pp.neighbors(n_neighbors=10, n_pcs=n_pcs)
nsclc_filtered <- FindNeighbors(nsclc_filtered, reduction = "pca", dims = 1:n_pcs, k.param = 10)

# Scanpy: sc.tl.louvain(resolution=..., random_state=198716)
# algorithm = 1 is the original Louvain algorithm (Seurat clusters the shared nearest neighbour graph)
for (res in c(0.5, 0.7)) {
  nsclc_filtered <- FindClusters(nsclc_filtered, resolution = res, algorithm = 1,
                                 random.seed = seed, cluster.name = paste0("preInt_louvain_", res))
}

# FindClusters makes the latest clustering the active identity; switch back to resolution 0.5
Idents(nsclc_filtered) <- "preInt_louvain_0.5"

c(clusters_preInt_louvain_0.5 = nlevels(nsclc_filtered$preInt_louvain_0.5),
  clusters_preInt_louvain_0.7 = nlevels(nsclc_filtered$preInt_louvain_0.7))

In [ ]:
# Scanpy: sc.tl.umap (min_dist = 0.5, neighbours from the graph above)
nsclc_filtered <- RunUMAP(nsclc_filtered, reduction = "pca", dims = 1:n_pcs, n.neighbors = 10, min.dist = 0.5,
                          seed.use = seed, reduction.name = "umap.preInt", reduction.key = "preIntUMAP_",
                          verbose = FALSE)
nsclc_filtered

##### UMAP coloured by sample

This is the main pre-Harmony view. If cells of the same type from different samples sit in separate islands, that is the batch effect Harmony should remove. Cells are drawn in random order (`shuffle = TRUE`) so that no sample hides the others.

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 5.5)
p <- DimPlot(nsclc_filtered, reduction = "umap.preInt", group.by = "sample", shuffle = TRUE, seed = seed) +
  ggtitle("Samples (without Harmony)")
save_plot(p, "11_umap_preInt_sample.png", width = 7, height = 5.5)

In [ ]:
# One panel per sample, to see where each sample's cells fall
options(repr.plot.width = 18, repr.plot.height = 5)
p <- DimPlot(nsclc_filtered, reduction = "umap.preInt", group.by = "sample", split.by = "sample", ncol = 4) +
  NoLegend() + ggtitle("Samples (without Harmony), one panel per sample")
save_plot(p, "12_umap_preInt_sample_split.png", width = 18, height = 5)

In [ ]:
# Sample next to the two clusterings
options(repr.plot.width = 18, repr.plot.height = 5)
p <- DimPlot(nsclc_filtered, reduction = "umap.preInt",
             group.by = c("sample", "preInt_louvain_0.5", "preInt_louvain_0.7"),
             ncol = 3, label = TRUE, repel = TRUE, shuffle = TRUE, seed = seed) & NoAxes()
save_plot(p, "13_umap_preInt_sample_clusters.png", width = 18, height = 5)

In [ ]:
# lets create groups based on QC violin plots and view them in UMAP (same cut-offs as stage 1; adjust after the violins above)
mt_cutoff   <- 5
ribo_cutoff <- 25
n_genes_low <- 2000

add_qc_groups <- function(obj) {
  obj$mt_group      <- ifelse(obj$percent.mt   < mt_cutoff,   "low_mt",      "high_mt")
  obj$ribo_group    <- ifelse(obj$percent.ribo < ribo_cutoff, "low_ribo",    "high_ribo")
  obj$n_genes_group <- ifelse(obj$nFeature_RNA < n_genes_low, "low_n_genes", "high_n_genes")
  obj
}

nsclc_filtered <- add_qc_groups(nsclc_filtered)

options(repr.plot.width = 16, repr.plot.height = 5)
p <- DimPlot(nsclc_filtered, reduction = "umap.preInt",
             group.by = c("mt_group", "ribo_group", "n_genes_group"), ncol = 3) & NoAxes()
save_plot(p, "14_umap_preInt_qc_groups.png", width = 16, height = 5)

In [ ]:
# Main compartments on the UMAP (RNA): immune (PTPRC), epithelial / tumour (EPCAM), fibroblasts (COL1A1), endothelial (PECAM1)
compartment_genes <- intersect(c("PTPRC", "EPCAM", "COL1A1", "PECAM1"), rownames(nsclc_filtered))

options(repr.plot.width = 10, repr.plot.height = 9)
p <- FeaturePlot(nsclc_filtered, features = compartment_genes, reduction = "umap.preInt", ncol = 2, order = TRUE) & NoAxes()
save_plot(p, "15_umap_preInt_compartment_markers.png", width = 10, height = 9)

In [ ]:
# TotalSeq antibodies on the UMAP (CLR-normalised ADT); "adt_" selects the ADT assay rather than the gene of the same name
adt_features <- paste0(Key(nsclc_filtered[["ADT"]]), rownames(nsclc_filtered[["ADT"]]))

options(repr.plot.width = 15, repr.plot.height = 13)
p <- FeaturePlot(nsclc_filtered, features = adt_features, reduction = "umap.preInt", ncol = 3, order = TRUE) & NoAxes()
save_plot(p, "16_umap_preInt_adt.png", width = 15, height = 13)

##### Sample make-up of each cluster (without Harmony)

This is the baseline to compare against after Harmony. Clusters made almost entirely of one sample are either batch effects or genuinely sample-specific cells. Malignant epithelial cells are often patient-specific because each tumour has its own copy-number changes. Immune and stromal clusters should mix across samples once the batch effect is removed.

In [ ]:
cluster_by_sample <- table(cluster = nsclc_filtered$preInt_louvain_0.5, sample = nsclc_filtered$sample)
write.csv(as.data.frame.matrix(cluster_by_sample), file.path(table_dir, "nsclc7pt_preInt_cluster_by_sample.csv"))
cluster_by_sample

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 5)
p <- ggplot(nsclc_filtered@meta.data, aes(x = preInt_louvain_0.5, fill = sample)) +
  geom_bar(position = "fill") +
  labs(title = "Sample composition per cluster (without Harmony)",
       x = "Cluster (preInt_louvain_0.5)", y = "Fraction of cells", fill = "Sample") +
  theme_classic()
save_plot(p, "17_cluster_sample_composition_preInt.png", width = 10, height = 5)

In [ ]:
saveRDS(nsclc_filtered, file = rds_file)
write.csv(nsclc_filtered@meta.data, file.path(table_dir, "nsclc7pt_preInt_cell_metadata.csv"))

#### Next: Harmony

`data/nsclc7pt_preInt.rds` holds all genes, raw counts (`counts`), log-normalised data (`data`), the PCA, the clusters and UMAP computed without Harmony, the `sample` label, and the number of PCs (`@misc$n_pcs`).

In [ ]:
sessionInfo()